In [20]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split,  GridSearchCV, KFold
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import cohen_kappa_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

gpu_devices = tf.config.experimental.list_physical_devices('GPU')
for device in gpu_devices:
    tf.config.experimental.set_memory_growth(device, True)
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
gpu_devices

def get_pred_report(y_test, y_pred):
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Accuracy Score: ", accuracy_score(y_test, y_pred))
    print("Cohen Kappa Score: ", cohen_kappa_score(y_test, y_pred))
    print("F1 Score: ", f1_score(y_test, y_pred))

In [24]:
df = pd.read_csv('features.csv')
X = df.drop('target', axis=1)
y = df['target']

In [25]:
X = X.drop(['max_val', 'min_val', 'sampen'], axis=1)
X.columns

Index(['amplitude', 'mean', 'variance', 'skewness', 'kurt', 'shannon', 'apen',
       'H', 'mmd', 'delta', 'theta', 'alpha', 'beta', 'gamma'],
      dtype='object')

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=8)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
get_pred_report(y_test, y_pred)


Confusion Matrix:
[[19416  1666]
 [ 2206 29063]]
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.92      0.91     21082
           1       0.95      0.93      0.94     31269

    accuracy                           0.93     52351
   macro avg       0.92      0.93      0.92     52351
weighted avg       0.93      0.93      0.93     52351

Accuracy Score:  0.9260377070161029
Cohen Kappa Score:  0.8468925723988656
F1 Score:  0.9375463724636278


In [26]:
X_new = X[['amplitude']]
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=8)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
get_pred_report(y_test, y_pred)

Confusion Matrix:
[[17423  3659]
 [ 5219 26050]]
Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.83      0.80     21082
           1       0.88      0.83      0.85     31269

    accuracy                           0.83     52351
   macro avg       0.82      0.83      0.83     52351
weighted avg       0.83      0.83      0.83     52351

Accuracy Score:  0.8304139366965292
Cohen Kappa Score:  0.6516780594095274
F1 Score:  0.8544065072649152


In [27]:
X_new = X[['mmd', 'alpha']]
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=8)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
get_pred_report(y_test, y_pred)

Confusion Matrix:
[[19252  1830]
 [ 3243 28026]]
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.91      0.88     21082
           1       0.94      0.90      0.92     31269

    accuracy                           0.90     52351
   macro avg       0.90      0.90      0.90     52351
weighted avg       0.91      0.90      0.90     52351

Accuracy Score:  0.903096406945426
Cohen Kappa Score:  0.8007408698420778
F1 Score:  0.9170061349693251


In [17]:
X_new = X[['mmd', 'alpha']]
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=8)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
get_pred_report(y_test, y_pred)

Confusion Matrix:
[[19252  1830]
 [ 3243 28026]]
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.91      0.88     21082
           1       0.94      0.90      0.92     31269

    accuracy                           0.90     52351
   macro avg       0.90      0.90      0.90     52351
weighted avg       0.91      0.90      0.90     52351

Accuracy Score:  0.903096406945426
Cohen Kappa Score:  0.8007408698420778
F1 Score:  0.9170061349693251


In [29]:
from os.path import join
import os
import numpy as np
import pandas as pd
import utils
ROOT_TEST_PATH = join("..","Data","test")
test_data = {i:np.load(join(ROOT_TEST_PATH, f"data_{i}.npy")) for i in [4,5]}

test_channels = [4,5]
test_data = None
for test_channel in test_channels:
    if os.path.exists(join(f"test_features.csv")):
        df_test = pd.read_csv(join(f"test_features.csv"))
    else:
        test_data = {i:np.load(join(ROOT_TEST_PATH, f"data_{i}.npy")) for i in [4,5]}
        break
    
    
if test_data is not None:
    df_list = []
    for record_number, data in test_data.items():
        data_filtered, data_raw = utils.process_data(data[:, :])
        for channel in range(data_filtered.shape[0]):
            df_sub_features = utils.compute_features_on_record(data_raw[channel], data_filtered[channel])
            df_sub_features['channel'] = channel
            df_sub_features['record_number'] = record_number
            df_list.append(df_sub_features)
            print(f"Record {record_number} Channel {channel} processed")
    
    df = pd.concat(df_list, ignore_index=True, axis=0)
    df.to_csv("test_features.csv", index=False)
    df_test = df

In [32]:
X_test = df_test.drop(['record_number', 'channel'], axis=1).copy()
#X_test = X_test.drop(['max_val', 'min_val', 'sampen'], axis=1)
X_test = X_test[['mmd', 'alpha']]

#y_pred = bst.predict(xgb.DMatrix(X_test))
y_pred = knn.predict(X_test)
y_pred = [round(value) for value in y_pred]
df_test['target'] = y_pred

df_test['count'] = df_test.groupby(['record_number', 'channel']).cumcount()
df_test['identifier'] = (df_test['record_number'] * 1e6 + (df_test['channel'] + 1) * 1e5 + df_test['count']).astype(int)
submission_df = df_test[["identifier", "target"]].copy()
submission_path = join("..", "Results", "submission_martin.csv")
submission_df.to_csv(submission_path, index=False)